# Translate-then-Summarize Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Translate-then-Summarize baseline for English-to-Chinese cross-lingual dialogue summarization.

The TS pipeline uses two local small language model agents. Agent 1 reads the original English dialogue and translates the full dialogue into Chinese while preserving speaker names, turn order, placeholders, and dialogue structure. Agent 2 then reads the translated Chinese dialogue and generates a concise Chinese summary.

```text
English Dialogue
→ Agent 1: Chinese Translation Agent
→ Agent 2: Chinese Summarization Agent
→ Final Chinese Summary
```

The pipeline consists of two agents:

```text
Agent 1: Chinese Translation Agent
Input: original English dialogue
Output: translated Chinese dialogue

Agent 2: Chinese Summarization Agent
Input: translated Chinese dialogue from Agent 1
Output: final Chinese summary
```
This setup is used as a Translate-then-Summarize baseline. Unlike the Direct pipeline, TS explicitly creates an intermediate Chinese dialogue translation before summarization. This allows us to inspect whether errors come from the translation stage or the summarization stage.

The local small language model is served through Ollama. The notebook controls the prompt design, agent workflow, input/output processing, intermediate output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the local model used for the Translate-then-Summarize baseline.

### Recommended model setup

This notebook uses one local small language model through Ollama for both agents:

- Agent 1: Chinese Translation Agent
- Agent 2: Chinese Summarization Agent

```bash
ollama pull qwen3.5:27b
```

If qwen3.5:27b is too slow on your machine, you can use a smaller model for testing:

```bash
ollama pull "qwen3.5:9b"
```

Make sure the model names in the notebook match the models installed in Ollama:

```bash
TRANSLATION_MODEL = "qwen3.5:27b"
SUMMARIZATION_MODEL = "qwen3.5:27b"
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [1]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# TS models
TRANSLATION_MODEL = "qwen3.5:27b"
SUMMARIZATION_MODEL = "qwen3.5:27b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Gold set path
GOLD_SET_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for TS baseline
FULL_OUTPUT_PATH = OUTPUT_DIR / "ts_qwen27b_50samples.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "ts_qwen27b_50samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "ts_qwen27b_50samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Gold set path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_50samples.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_50samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/ts_qwen27b_50samples_errors.jsonl


In [3]:
print(GOLD_SET_PATH.exists())

True


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['gemma3:27b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: TS prompt templates

CHINESE_TRANSLATION_PROMPT = """You are a dialogue translation agent.

Your task is to translate the following English dialogue into Chinese.

Requirements:
- Translate the full dialogue into Chinese.
- Preserve the speaker names.
- Preserve the dialogue structure and turn order.
- Preserve placeholders such as <file_photo>, <file_other>, <location>, or emojis if they appear.
- Do not summarize the dialogue.
- Do not omit any important information.
- Do not add information that is not in the original dialogue.
- Output only the translated Chinese dialogue.

STRICT TRANSLATION RULE:
- You MUST translate ALL English proper nouns and speaker names into standard Chinese characters.
- ABSOLUTELY NO English letters or names should appear in the final Chinese summary.

English dialogue:
{dialogue}

Chinese translation:
"""


CHINESE_SUMMARIZATION_PROMPT = """You are a Chinese dialogue summarization agent.

Your task is to read the translated Chinese dialogue and generate a concise Chinese summary.

Requirements:
- Summarize the main information in the dialogue.
- Write the summary in Chinese.
- Keep the summary concise and faithful to the dialogue.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the final Chinese summary.

Conciseness: 
- For simple dialogues, prefer 20-50 Chinese characters. 
- For complex dialogues, allow up to 80 Chinese characters.

Chinese dialogue:
{translated_dialogue}

Chinese summary:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: TS agent functions

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def chinese_translation_agent(dialogue: str) -> str:
    """Agent 1: English dialogue -> translated Chinese dialogue."""
    prompt = fill_prompt(
        CHINESE_TRANSLATION_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()


def chinese_summarization_agent(translated_dialogue: str) -> str:
    """Agent 2: translated Chinese dialogue -> Chinese summary."""
    prompt = fill_prompt(
        CHINESE_SUMMARIZATION_PROMPT,
        {
            "translated_dialogue": translated_dialogue,
        },
    )

    response = call_ollama(
        model=SUMMARIZATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: TS pipeline

def run_ts_pipeline(example: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the Translate-then-Summarize pipeline."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: English dialogue -> translated Chinese dialogue
    translated_dialogue = chinese_translation_agent(dialogue)

    if verbose:
        print("\n" + "=" * 80)
        print(f"Sample ID: {sample_id}")
        print("=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===")
        print(translated_dialogue)
        print("=" * 80 + "\n")

    # Agent 2: translated Chinese dialogue -> Chinese summary
    final_chinese_summary = chinese_summarization_agent(translated_dialogue)

    if verbose:
        print("=== Agent 2 Final Output: Chinese Summary ===")
        print(final_chinese_summary)
        print("=" * 80 + "\n")

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,

        # Intermediate output from Agent 1
        "translated_dialogue": translated_dialogue,

        # Final output from Agent 2
        "final_summary": final_chinese_summary,

        # References
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        # Metadata
        "pipeline": "translate_then_summarize",
        "translation_model": TRANSLATION_MODEL,
        "summarization_model": SUMMARIZATION_MODEL,
        "num_model_calls": 2,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect whether errors come from the English summarization stage or the Chinese translation stage.

In [10]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=50)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 50 examples.
First example:
{'id': 'gold_00001', 'test_index': 23, 'dialogue': "Anne: You were right, he was lying to me :/\nIrene: Oh no, what happened?\nJane: who? that Mark guy?\nAnne: yeah, he told me he's 30, today I saw his passport - he's 40\nIrene: You sure it's so important?\nAnne: he lied to me Irene", 'reference_english_summary': 'Mark lied to Anne about his age. Mark is 40.', 'reference_chinese_summary': '马克向安妮隐瞒了自己的年龄。他40岁了。'}


In [11]:
# Cell 10: Run the TS pipeline for the first example

result = run_ts_pipeline(test_data[4], verbose=True)
result


Sample ID: gold_00005
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
乔伊斯：快来看这个！
乔伊斯：<link>
迈克尔：太便宜了！
埃德森：不可能！我现在就订票！！

=== Agent 2 Final Output: Chinese Summary ===
乔伊斯分享了一个链接，迈克尔感叹价格太便宜，埃德森则立即决定订票。



{'id': 'gold_00005',
 'test_index': 66,
 'dialogue': "Joyce: Check this out!\r\nJoyce: <link>\r\nMichael: That's cheap!\r\nEdson: No way! I'm booking my ticket now!! ",
 'translated_dialogue': '乔伊斯：快来看这个！\n乔伊斯：<link>\n迈克尔：太便宜了！\n埃德森：不可能！我现在就订票！！',
 'final_summary': '乔伊斯分享了一个链接，迈克尔感叹价格太便宜，埃德森则立即决定订票。',
 'reference_english_summary': 'Edson is booking his ticket now.',
 'reference_chinese_summary': '埃德森正在订票。',
 'pipeline': 'translate_then_summarize',
 'translation_model': 'qwen3.5:27b',
 'summarization_model': 'qwen3.5:27b',
 'num_model_calls': 2}

In [12]:
# Cell 11: Print TS pipeline result clearly

def print_ts_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===")
    print(result["translated_dialogue"])
    print()

    print("=== Agent 2 Final Output: Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Translation model:", result["translation_model"])
    print("Summarization model:", result["summarization_model"])
    print("Model calls:", result["num_model_calls"])


print_ts_result(result)

=== Original Dialogue ===
Joyce: Check this out!
Joyce: <link>
Michael: That's cheap!
Edson: No way! I'm booking my ticket now!! 

=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
乔伊斯：快来看这个！
乔伊斯：<link>
迈克尔：太便宜了！
埃德森：不可能！我现在就订票！！

=== Agent 2 Final Output: Chinese Summary ===
乔伊斯分享了一个链接，迈克尔感叹价格太便宜，埃德森则立即决定订票。

=== Reference English Summary ===
Edson is booking his ticket now.

=== Reference Chinese Summary ===
埃德森正在订票。

=== Metadata ===
Pipeline: translate_then_summarize
Translation model: qwen3.5:27b
Summarization model: qwen3.5:27b
Model calls: 2


## 4. Save Results

This saves the intermediate English summary and the final Chinese summary.

In [13]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\ts_qwen27b_50samples.jsonl
CSV output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\ts_qwen27b_50samples.csv
Error output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\ts_qwen27b_50samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed ST result to the JSONL output file.

If the notebook stops, already processed examples remain saved.

In [14]:
# Cell 13: Batch inference with TS pipeline
# Time stamp: 2m 51.7s

MAX_EXAMPLES = 50
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running TS pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        print(f"Skipping already processed sample: {sample_id}")
        continue

    try:
        record = run_ts_pipeline(ex, verbose=True)

        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running TS pipeline:   0%|          | 0/50 [00:00<?, ?it/s]


Sample ID: gold_00001
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
安妮：你是对的，他对我撒谎了 :/
艾琳：哦不，发生什么事了？
简：谁？那个马克吗？
安妮：是的，他告诉我他三十岁，今天我看到了他的护照——他四十岁。
艾琳：你确定这很重要吗？
安妮：他对我撒谎了，艾琳。

=== Agent 2 Final Output: Chinese Summary ===
安妮发现马克在年龄上对她撒谎，他自称三十岁，但护照显示实际为四十岁。尽管艾琳质疑此事的重要性，安妮仍坚持认为被欺骗很严重。


Sample ID: gold_00002
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
玛丽：嘿，我手头有点紧，借我几盒吧。
卡特：好的，给我一小时，我在火车站。
玛丽：太好了，谢谢。

=== Agent 2 Final Output: Chinese Summary ===
玛丽因手头紧向卡特借几盒，卡特答应一小时后在火车站交付，玛丽表示感谢。


Sample ID: gold_00003
=== Agent 1 Intermediate Output: Translated Chinese Dialogue ===
蒂娜：我告诉你，这家阿联酋航空的机组人员看起来太棒了，就像电影明星一样。
阿拉：哦，是的，我知道，这是有目的的。
阿拉：本来就应该这样。
阿拉：他们非常注重形象。
蒂娜：看起来不错，看着真让人愉悦。
蒂娜：我在机场可倒霉了，他们在飞机上把我们滞留了一个小时，最后我才赶上回家的晚班航班。
蒂娜：你能想象吗？
蒂娜：而且你知道吗，这次我们遇到了一位特别健谈的飞行员 :-)
阿拉：哦，可怜的你。
阿拉：哼。
阿拉：我现在正要去开会。
蒂娜：就是那个会议吗？
阿拉：是的，祝你好运。
蒂娜：当然，告诉我结果如何。
阿拉：好的，亲爱的，保持联系。

=== Agent 2 Final Output: Chinese Summary ===
蒂娜称赞阿联酋航空机组形象，但抱怨机场滞留及航班延误。阿拉表示同情，随后告知要去开会，两人约定保持联系。


Sample ID: gold_0

## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.

For the ST pipeline, the CSV also includes the intermediate English summary from Agent 1.

In [15]:
# Cell 14: Export TS summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),

        # Intermediate output from Agent 1
        "translated_dialogue": record.get("translated_dialogue", ""),

        # Final output from Agent 2
        "final_summary": record.get("final_summary", ""),

        # References
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        # Metadata
        "pipeline": record.get("pipeline", "translate_then_summarize"),
        "translation_model": record.get("translation_model", ""),
        "summarization_model": record.get("summarization_model", ""),
        "num_model_calls": record.get("num_model_calls", 2),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\ts_qwen27b_50samples.csv


,id,test_index,dialogue,translated_dialogue,final_summary,reference_english_summary,reference_chinese_summary,pipeline,translation_model,summarization_model,num_model_calls
0,gold_00001,23,"Anne: You were right, he was lying to me :/\nI...",安妮：你是对的，他对我撒谎了 :/\n艾琳：哦不，发生什么事了？\n简：谁？那个马克吗？\n...,安妮发现马克在年龄上对她撒谎，他自称三十岁，但护照显示实际为四十岁。尽管艾琳质疑此事的重要性...,Mark lied to Anne about his age. Mark is 40.,马克向安妮隐瞒了自己的年龄。他40岁了。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
1,gold_00002,30,"Mary: hey, im kinda broke, lend me a few box\r...",玛丽：嘿，我手头有点紧，借我几盒吧。\n卡特：好的，给我一小时，我在火车站。\n玛丽：太好了...,玛丽因手头紧向卡特借几盒，卡特答应一小时后在火车站交付，玛丽表示感谢。,Mary ran out of money. Carter is going to lend...,玛丽的钱用完了，卡特打算一小时后借给她一点。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
2,gold_00003,39,"Tina: I'll tell you something, this Emirate st...",蒂娜：我告诉你，这家阿联酋航空的机组人员看起来太棒了，就像电影明星一样。\n阿拉：哦，是的，...,蒂娜称赞阿联酋航空机组形象，但抱怨机场滞留及航班延误。阿拉表示同情，随后告知要去开会，两人约...,Tina will catch the evening flight back home. ...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
3,gold_00004,65,Ana: You sleeping?\r\nCatherine: Not yet.\r\nA...,安娜：你睡着了吗？\n凯瑟琳：还没。\n安娜：明天想去探望奶奶吗？我想她了。\n凯瑟琳：好啊...,安娜提议明天探望奶奶，凯瑟琳欣然同意，并约定醒来后联系，随后两人互道晚安。,Ana wants to visit grandma tomorrow. Catherine...,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
4,gold_00005,66,Joyce: Check this out!\r\nJoyce: <link>\r\nMic...,乔伊斯：快看看这个！\n乔伊斯：<link>\n迈克尔：太便宜了！\n埃德森：不可能！我现在...,乔伊斯分享了一个链接，迈克尔感叹价格太便宜，埃德森则决定立即订票。,Edson is booking his ticket now.,埃德森正在订票。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
5,gold_00006,67,Jane: google maps says it is at least 3h <file...,简：谷歌地图显示至少需要 3 小时 <file_other>\n史蒂文：我以前 2 小时就到...,简提议将见面时间改为 4 点半，史蒂文同意。两人确认将在主入口见面。,Jane wants to leave at 4.30 instead of 5 becau...,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
6,gold_00007,78,"Fiona: Are you free?\r\nTina: Yes, what's up?\...",菲奥娜：你有空吗？\n蒂娜：有空，怎么了？\n菲奥娜：我正在为克里斯准备一顿丰盛的晚餐，我在...,菲奥娜邀请蒂娜帮忙为克里斯做挞。菲奥娜已买好挞皮，但馅料做成了柠檬炒蛋。蒂娜安慰她，并指出加...,Fiona wants to prepare dinner for Chris. She i...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
7,gold_00008,86,Olafur: are we doing anything for New Year's E...,奥拉夫：我们除夕夜有什么安排吗？\n娜塔莉：我在想一些高雅的活动，比如歌剧之类的。\n佐伊：...,三人讨论除夕安排，娜塔莉提议高雅活动，奥拉夫建议派对。最终他们选定苏荷区一家俱乐部，并决定立...,"Nathalie, Olafur and Zoe are planning the New ...",娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
8,gold_00009,120,"John: wanna go see ""A Star is Born"" on Wed?\r\...",约翰：周三想看《一个明星的诞生》吗？\n琼：抱歉，不行。\n琼：超级忙。\n琼：没时间做任何...,约翰邀请琼周三看电影被拒，因琼太忙。两人最终约定周四晚八点观看《一个明星的诞生》，约翰将查询...,"Joan and John are going to watch ""A Star is Bo...",琼和约翰星期四晚上8点左右去看《一个明星的诞生》。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2
9,gold_00010,137,Peyton: I have been asking you to bring that v...,佩顿：我一直让你帮我带那个电子游戏。\n卡梅伦：亲爱的，我实在没有足够的时间回家。\n佩顿：...,佩顿催促卡梅伦带游戏回家，卡梅伦因在外地需停留一周无法带回。佩顿建议快递，卡梅伦抱怨其刻薄，...,Peyton is expecting Cameron to bring the video...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。,translate_then_summarize,qwen3.5:27b,qwen3.5:27b,2


In [16]:
# Cell 15: Compare TS outputs with references

comparison_columns = [
    "id",
    "test_index",
    "translated_dialogue",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,test_index,translated_dialogue,final_summary,reference_chinese_summary
0,gold_00001,23,安妮：你是对的，他对我撒谎了 :/\n艾琳：哦不，发生什么事了？\n简：谁？那个马克吗？\n...,安妮发现马克在年龄上对她撒谎，他自称三十岁，但护照显示实际为四十岁。尽管艾琳质疑此事的重要性...,马克向安妮隐瞒了自己的年龄。他40岁了。
1,gold_00002,30,玛丽：嘿，我手头有点紧，借我几盒吧。\n卡特：好的，给我一小时，我在火车站。\n玛丽：太好了...,玛丽因手头紧向卡特借几盒，卡特答应一小时后在火车站交付，玛丽表示感谢。,玛丽的钱用完了，卡特打算一小时后借给她一点。
2,gold_00003,39,蒂娜：我告诉你，这家阿联酋航空的机组人员看起来太棒了，就像电影明星一样。\n阿拉：哦，是的，...,蒂娜称赞阿联酋航空机组形象，但抱怨机场滞留及航班延误。阿拉表示同情，随后告知要去开会，两人约...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。
3,gold_00004,65,安娜：你睡着了吗？\n凯瑟琳：还没。\n安娜：明天想去探望奶奶吗？我想她了。\n凯瑟琳：好啊...,安娜提议明天探望奶奶，凯瑟琳欣然同意，并约定醒来后联系，随后两人互道晚安。,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。
4,gold_00005,66,乔伊斯：快看看这个！\n乔伊斯：<link>\n迈克尔：太便宜了！\n埃德森：不可能！我现在...,乔伊斯分享了一个链接，迈克尔感叹价格太便宜，埃德森则决定立即订票。,埃德森正在订票。
5,gold_00006,67,简：谷歌地图显示至少需要 3 小时 <file_other>\n史蒂文：我以前 2 小时就到...,简提议将见面时间改为 4 点半，史蒂文同意。两人确认将在主入口见面。,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...
6,gold_00007,78,菲奥娜：你有空吗？\n蒂娜：有空，怎么了？\n菲奥娜：我正在为克里斯准备一顿丰盛的晚餐，我在...,菲奥娜邀请蒂娜帮忙为克里斯做挞。菲奥娜已买好挞皮，但馅料做成了柠檬炒蛋。蒂娜安慰她，并指出加...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。
7,gold_00008,86,奥拉夫：我们除夕夜有什么安排吗？\n娜塔莉：我在想一些高雅的活动，比如歌剧之类的。\n佐伊：...,三人讨论除夕安排，娜塔莉提议高雅活动，奥拉夫建议派对。最终他们选定苏荷区一家俱乐部，并决定立...,娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...
8,gold_00009,120,约翰：周三想看《一个明星的诞生》吗？\n琼：抱歉，不行。\n琼：超级忙。\n琼：没时间做任何...,约翰邀请琼周三看电影被拒，因琼太忙。两人最终约定周四晚八点观看《一个明星的诞生》，约翰将查询...,琼和约翰星期四晚上8点左右去看《一个明星的诞生》。
9,gold_00010,137,佩顿：我一直让你帮我带那个电子游戏。\n卡梅伦：亲爱的，我实在没有足够的时间回家。\n佩顿：...,佩顿催促卡梅伦带游戏回家，卡梅伦因在外地需停留一周无法带回。佩顿建议快递，卡梅伦抱怨其刻薄，...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。


In [17]:
# Cell 16: Inspect TS outputs

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "translated_dialogue",
        "final_summary",
        "reference_english_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,translated_dialogue,final_summary,reference_english_summary,reference_chinese_summary
0,gold_00001,23,"Anne: You were right, he was lying to me :/\nI...",安妮：你是对的，他对我撒谎了 :/\n艾琳：哦不，发生什么事了？\n简：谁？那个马克吗？\n...,安妮发现马克在年龄上对她撒谎，他自称三十岁，但护照显示实际为四十岁。尽管艾琳质疑此事的重要性...,Mark lied to Anne about his age. Mark is 40.,马克向安妮隐瞒了自己的年龄。他40岁了。
1,gold_00002,30,"Mary: hey, im kinda broke, lend me a few box\r...",玛丽：嘿，我手头有点紧，借我几盒吧。\n卡特：好的，给我一小时，我在火车站。\n玛丽：太好了...,玛丽因手头紧向卡特借几盒，卡特答应一小时后在火车站交付，玛丽表示感谢。,Mary ran out of money. Carter is going to lend...,玛丽的钱用完了，卡特打算一小时后借给她一点。
2,gold_00003,39,"Tina: I'll tell you something, this Emirate st...",蒂娜：我告诉你，这家阿联酋航空的机组人员看起来太棒了，就像电影明星一样。\n阿拉：哦，是的，...,蒂娜称赞阿联酋航空机组形象，但抱怨机场滞留及航班延误。阿拉表示同情，随后告知要去开会，两人约...,Tina will catch the evening flight back home. ...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。
3,gold_00004,65,Ana: You sleeping?\r\nCatherine: Not yet.\r\nA...,安娜：你睡着了吗？\n凯瑟琳：还没。\n安娜：明天想去探望奶奶吗？我想她了。\n凯瑟琳：好啊...,安娜提议明天探望奶奶，凯瑟琳欣然同意，并约定醒来后联系，随后两人互道晚安。,Ana wants to visit grandma tomorrow. Catherine...,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。
4,gold_00005,66,Joyce: Check this out!\r\nJoyce: <link>\r\nMic...,乔伊斯：快看看这个！\n乔伊斯：<link>\n迈克尔：太便宜了！\n埃德森：不可能！我现在...,乔伊斯分享了一个链接，迈克尔感叹价格太便宜，埃德森则决定立即订票。,Edson is booking his ticket now.,埃德森正在订票。
5,gold_00006,67,Jane: google maps says it is at least 3h <file...,简：谷歌地图显示至少需要 3 小时 <file_other>\n史蒂文：我以前 2 小时就到...,简提议将见面时间改为 4 点半，史蒂文同意。两人确认将在主入口见面。,Jane wants to leave at 4.30 instead of 5 becau...,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...
6,gold_00007,78,"Fiona: Are you free?\r\nTina: Yes, what's up?\...",菲奥娜：你有空吗？\n蒂娜：有空，怎么了？\n菲奥娜：我正在为克里斯准备一顿丰盛的晚餐，我在...,菲奥娜邀请蒂娜帮忙为克里斯做挞。菲奥娜已买好挞皮，但馅料做成了柠檬炒蛋。蒂娜安慰她，并指出加...,Fiona wants to prepare dinner for Chris. She i...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。
7,gold_00008,86,Olafur: are we doing anything for New Year's E...,奥拉夫：我们除夕夜有什么安排吗？\n娜塔莉：我在想一些高雅的活动，比如歌剧之类的。\n佐伊：...,三人讨论除夕安排，娜塔莉提议高雅活动，奥拉夫建议派对。最终他们选定苏荷区一家俱乐部，并决定立...,"Nathalie, Olafur and Zoe are planning the New ...",娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...
8,gold_00009,120,"John: wanna go see ""A Star is Born"" on Wed?\r\...",约翰：周三想看《一个明星的诞生》吗？\n琼：抱歉，不行。\n琼：超级忙。\n琼：没时间做任何...,约翰邀请琼周三看电影被拒，因琼太忙。两人最终约定周四晚八点观看《一个明星的诞生》，约翰将查询...,"Joan and John are going to watch ""A Star is Bo...",琼和约翰星期四晚上8点左右去看《一个明星的诞生》。
9,gold_00010,137,Peyton: I have been asking you to bring that v...,佩顿：我一直让你帮我带那个电子游戏。\n卡梅伦：亲爱的，我实在没有足够的时间回家。\n佩顿：...,佩顿催促卡梅伦带游戏回家，卡梅伦因在外地需停留一周无法带回。佩顿建议快递，卡梅伦抱怨其刻薄，...,Peyton is expecting Cameron to bring the video...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。
